
# 🧮 Week 2 — How LLMs Work: Probability, Sampling & Prompts

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/lectures/probability_lecture.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

📘 **Theme:** Understanding LLMs as probabilistic systems — linking mathematical intuition to model behavior and creative control.

---

### **Learning Objectives**
By the end of this week, you will be able to:
1. Write the probabilistic formulation of an LLM.
2. Explain how maximum likelihood training shapes model behavior.
3. Describe how temperature and entropy control randomness.
4. Analyze how prompts condition outputs.
5. Relate math intuition to real-world UX reliability.


In [ ]:
# @title Setup (Run this first)
!git clone --depth 1 -q https://github.com/tulane-intro-ai-engineering/main.git
import sys; sys.path.append('/content/main')
from course_utils import lab2_setup, show_mermaid

lab2_setup()
print("✅ Environment ready!")

Enter your OpenAI API key. It will only live in this Colab runtime.
OpenAI API key: ··········
✅ API key set.
✅ lab2_setup complete — scientific libraries ready, helper function loaded.
✅ Environment ready!



## 🧩 Day 1 — A Simple Model of an LLM
---
**Guiding Question:**  
> What does it mean to say that an LLM “predicts the next token”?  
> How do probabilities create creativity *and* hallucinations?



### 🔢 The LLM as a Probability Model

A language model defines the probability of a sequence as:

$$
P(x_1, x_2, ..., x_T) = \prod_{t=1}^{T} P(x_t \mid x_{1:t-1})
$$

Each word is drawn from a conditional distribution based on what came before.


In [ ]:
import numpy as np

tokens = ["is", "was", "will be", "seems"]
probs = np.array([0.65, 0.20, 0.10, 0.05])

for i in range(5):
    print(f"Sample {i+1}: {np.random.choice(tokens, p=probs)}")

Sample 1: is
Sample 2: seems
Sample 3: was
Sample 4: is
Sample 5: is



### 🧮 Maximum Likelihood Training

Training seeks parameters $\theta$ that make real text more probable:

$$
\theta^* = \arg\max_{\theta} \sum_{t=1}^{T} \log P_\theta(x_t \mid x_{1:t-1})
$$

Taking the log makes it easier to optimize (turns multiplication into addition).

In simple terms: the model rewards itself when it correctly predicts real text.


In [ ]:
true_next = "mat"
pred_probs = {"mat": 0.7, "rug": 0.2, "dog": 0.1}

import numpy as np
print("True token:", true_next)
print("Predicted probabilities:", pred_probs)
print("Log-likelihood contribution:", np.log(pred_probs[true_next]))

True token: mat
Predicted probabilities: {'mat': 0.7, 'rug': 0.2, 'dog': 0.1}
Log-likelihood contribution: -0.35667494393873245



### 🎭 Why Likelihood ≠ Truth

The model optimizes *linguistic probability*, not *factual accuracy*:

$$
P(\text{"2 + 2 = 4"}) \approx 0.99, \quad P(\text{"2 + 2 = 5"}) \approx 0.01
$$

For unseen prompts, it predicts what **sounds** likely — even if false.

> **Hallucination:** high $P(\text{text}|\text{context})$, low $P(\text{truth}|\text{world})$.



### 🧩 Unifying Diagram v1 — Adding Training Data Distribution

Everything the model knows comes from its **training data distribution**, which shapes its probabilities.



In [ ]:
# @title Updated LLM diagram with training data

show_mermaid(
    """
    graph TD
    subgraph User Interaction
    U["👤 Users<br/>Queries / Inputs"]:::user --> IH["Input Handling<br/>• Formatting<br/>• Validation<br/>• Safety Filters"]:::process
    end

    subgraph Prompt & Control
    IH --> PC("Prompt / Control<br/>• Instructions<br/>• Examples<br/>• Constraints<br/>• Temperature & Sampling"):::control
    end

    subgraph Tools & Augmentation
    PC --> TF{"Tools / Functions<br/>• External APIs"}:::tool
    PC --> RAG{"Retrieval (RAG)<br/>• Embeddings<br/>• Vector Store<br/>• Top-k Search"}:::tool
    end

    subgraph Core LLM
    TF --> LLM["Core LLM<br/>• Next-token probabilities<br/>• Sampling<br/>• Fine-tuned weights"]:::model
    RAG --> LLM
    PC --> LLM
    end

    subgraph Model Training
    D["📚 Training Data Distribution<br/>• Text corpus<br/>• Domain sources<br/>• Biases"]:::data --> LLM
    end

    subgraph Output & Monitoring
    LLM --> OP["Output Processing<br/>• Formatting<br/>• Citations<br/>• Refusals<br/>• Trust Signals"]:::output --> O("Final Output"):::output
    O --> LM["Logging & Monitoring<br/>• Prompts & Responses<br/>• Metrics<br/>• Drift Detection"]:::monitor
    end

    classDef user fill:#d1e7dd,stroke:#333,stroke-width:1px;
    classDef process fill:#e2e3e5,stroke:#333,stroke-width:1px;
    classDef control fill:#cfe2ff,stroke:#333,stroke-width:1px;
    classDef tool fill:#fff3cd,stroke:#333,stroke-width:1px;
    classDef model fill:#f8d7da,stroke:#333,stroke-width:1px;
    classDef output fill:#e9ecef,stroke:#333,stroke-width:1px;
    classDef monitor fill:#fefefe,stroke:#333,stroke-width:1px;
    classDef data fill:#fde2e4,stroke:#333,stroke-width:1px;
    """
)


### 🧩 Concept Check — Day 1

**Question:**  
Why does the model compute $P(x_t \mid x_{1:t-1})$ instead of predicting an entire sentence at once?


In [ ]:
# @title 💡 Show Suggested Answer
# Because predicting token by token allows the model to dynamically update
# its probability distribution as context grows.
# This incremental approach captures dependencies more flexibly and enables adaptive reasoning.



## 💻 Day 2 — Sampling, Temperature, and Prompt Patterns
---
**Guiding Question:**  
> How does randomness influence creativity and determinism in LLMs?



### 🌡️ Temperature and Softmax Sampling

Models sample from a softmax distribution:

$$
P_T(x_t=i) = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}
$$

- **Low T (e.g., 0.3):** sharper distribution → repetitive, factual.  
- **High T (e.g., 2.0):** flatter distribution → creative, variable.


In [ ]:
import numpy as np

def softmax_temp(z, T):
    p = np.exp(z / T)
    p /= p.sum()
    return p

logits = np.array([2.0, 1.0, 0.1])
for T in [0.3, 1.0, 2.0]:
    print(f"T={T}: {softmax_temp(logits, T)}")

T=0.3: [0.96390178 0.03438623 0.00171199]
T=1.0: [0.65900114 0.24243297 0.09856589]
T=2.0: [0.50168776 0.30428901 0.19402324]



### 🔀 Entropy: Measuring Uncertainty

Entropy quantifies unpredictability:

$$
H(P) = -\sum_i P_i \log_2 P_i
$$

- Low $H$ → confident, stable predictions  
- High $H$ → uncertain, diverse outputs


In [ ]:
import math

def entropy(p): return -sum(pi * math.log(pi, 2) for pi in p)

print("Entropy([0.9, 0.1]) =", entropy([0.9, 0.1]))
print("Entropy([0.5, 0.5]) =", entropy([0.5, 0.5]))

Entropy([0.9, 0.1]) = 0.4689955935892812
Entropy([0.5, 0.5]) = 1.0



### 🧠 Prompt Patterns and Conditioning

Prompts provide *context* that reshapes token probabilities:

$$
P_\theta(x_t \mid x_{1:t-1}, \text{prompt})
$$

Common patterns include:
1. **Role prompts:** “You are a teacher explaining entropy.”  
2. **Example prompts:** “Q: ... A: ...”  
3. **Constraint prompts:** “Answer in three bullet points.”



### 🧩 Concept Check — Prompt Conditioning

**Question:**  
How do prompts change a model’s behavior?

*(Hint: think in terms of conditional probability — what changes in $P_\theta(x_t \mid x_{1:t-1}, \text{prompt})$?)*


In [ ]:
# @title 💡 Show Suggested Answer
# Prompts alter the **conditioning context** the model uses when generating text.
# They effectively change the conditional probability distribution:
#     P(x_t | x_{1:t-1}, prompt)
# making the model sample from a distribution consistent with the prompt's tone, role, or constraints.



### 🧪 Bridge to Lab 2 — Measuring Diversity

In Lab 2, you will vary `temperature` and measure diversity:

$$
\text{Diversity Index} = \frac{\text{unique tokens}}{\text{total tokens}}
$$

You’ll test how higher temperature increases diversity but may reduce accuracy.



### 🧩 Concept Check — Day 2

**Question:**  
How does temperature affect the creativity and determinism of model outputs?


In [ ]:
# @title 💡 Show Suggested Answer
# Temperature controls the randomness of the model's sampling.
# Higher temperature → more random and creative outputs (higher entropy).
# Lower temperature → more deterministic and focused predictions.



### 💭 Reflection — Connecting Theory to Design

- When might you prefer **low entropy** (predictable) text?
- When might you prefer **high entropy** (creative) text?
- How does this relate to *reliability vs. creativity* in real-world AI systems?



<details>
<summary>🧑‍🏫 Instructor Notes</summary>

**Day 1 Focus:**  
- Emphasize intuition behind $P(x_t | x_{1:t-1})$.  
- Use visual examples (like bar charts) to show shifting probabilities.  

**Day 2 Focus:**  
- Encourage students to experiment interactively with temperature.  
- Discuss connections between entropy, creativity, and trustworthiness.  

**Extension Ideas:**  
- Demonstrate `top_p` sampling for comparison.  
- Optional: short visualization of entropy vs. creativity using matplotlib.
</details>
